# Solution R9 · Warm-up: the first XGBoost and early stopping

**Type**: Implement with scikit-learn

## Objective

Train an `XGBClassifier` with **early stopping** on a validation set, see how many rounds it really uses, compare it with the same model at a fixed 1000 rounds and with a forest.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(3000, n_features=12, n_informative=5, class_sep=0.8, flip_y=0.05, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)

## Commented solution

In [ ]:
model = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3, eval_metric="logloss",
                          early_stopping_rounds=30, n_jobs=-1, random_state=0)
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
rounds_used = int(model.best_iteration) + 1
p = model.predict_proba(X_te)[:, 1]
es_auc, es_ll = float(roc_auc_score(y_te, p)), float(log_loss(y_te, p))
print(f"rounds used: {rounds_used} out of 1000 | test AUC {es_auc:.4f} | test log-loss {es_ll:.4f}")

In [ ]:
no_es = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3, eval_metric="logloss", n_jobs=-1, random_state=0).fit(X_tr, y_tr)
ll_1000 = float(log_loss(y_te, no_es.predict_proba(X_te)[:, 1]))
print(f"without early stopping (1000 rounds): test log-loss {ll_1000:.4f} (with early stopping {es_ll:.4f})")

In [ ]:
rf = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
rf_auc = float(roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1]))
xgb_wins = bool(es_auc > rf_auc)
print(f"test AUC: XGBoost {es_auc:.4f} | forest {rf_auc:.4f} -> XGBoost wins: {xgb_wins}")

## Check

In [ ]:
with verify():
    check(50 < rounds_used < 500 and es_auc > 0.9, "early stopping halts between 50 and 500 rounds (out of 1000) and the AUC exceeds 0.9", "model.best_iteration + 1")
    check(es_ll < ll_1000 - 0.01, "with early stopping the test log-loss is better by over 0.01 than at 1000 rounds", "log_loss(y_te, no_es.predict_proba(X_te)[:, 1])")
    check(0.9 < rf_auc < 1.0 and xgb_wins in (True, False), "forest AUC between 0.9 and 1; `xgb_wins` is a boolean", "roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1])")
done()

## What you learned

- **Early stopping chooses the rounds for you:** the model stops at 286 rounds out of 1000. The high ceiling costs nothing because the training is interrupted when the validation log-loss stops dropping.
- **Not stopping costs:** at 1000 rounds the test log-loss is 0.357 against 0.334 with early stopping (+7%), while the AUC changes very little (0.929 against 0.932): the overfitting of boosting shows up in the probabilities before it shows in the classification (module 9, exercise C1).
- **Boosting does not always win:** on this data the forest with the defaults has AUC 0.941 against 0.932 for XGBoost. With noisy data (here 5% of random labels) boosting chases the noise and a forest is more robust; to beat the forest you need tuning (module 9, exercises C2 and F2).

## Common pitfall

Using the test set as `eval_set` for early stopping: you chose the number of rounds by looking at the test set. The validation set is a separate set, cut from the training data.

## Interview question

<details><summary>Why do you fix a high number of rounds in boosting and use early stopping?</summary>

Because the right number of rounds depends on the data and on the learning rate, and with too many rounds the model overfits (unlike a forest). You set a high ceiling, a low learning rate and let the training stop when the validation metric stops improving, then you evaluate once on the test set.

</details>